# Análisis Exploratorio de datos
---

**Autor:** Jaime Lozano Cillero

## 0. Configuración del Notebook
Importaremos todas las librerías y funciones que vemos relevantes para el notebook que vamos a crear.

### Importación de librerías

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
# import plotly.express as px
# import plotly.graph_objects as go

# from matplotlib.colors import ListedColormap

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    roc_curve,
    roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)

from typing import List, Optional, Tuple, Union

### Definición de constantes

In [2]:
PATH_DIRECTORIO_DATOS = "../../data"

In [7]:
PATH_DATASET_PRACTICA_FINAL = f"{PATH_DIRECTORIO_DATOS}/raw/dataset_practica_final.csv"

### Definición de funciones

In [4]:
# Calculamos las métricas de evaluación
def calcular_metricas_evaluacion(y_prediccion: np.ndarray, y_real: np.ndarray, verbose: bool = True):
    """Calcula las métricas de evaluación para un modelo de regresión.
    
    Calcula cuatro métricas comunes para evaluar modelos de regresión: MSE (Error Cuadrático Medio),
    RMSE (Raíz del Error Cuadrático Medio), MAE (Error Absoluto Medio) y R² (Coeficiente de determinación).
    Opcionalmente imprime los resultados en un formato legible.
    
    Args:
        y_prediccion (np.ndarray): Valores predichos por el modelo.
        y_real (np.ndarray): Valores reales observados.
        verbose (bool, optional): Si es True, imprime las métricas calculadas. Por defecto es True.
    
    Returns:
        tuple[float, float, float, float]: Una tupla con cuatro valores en el siguiente orden:
            - mse: Error cuadrático medio.
            - rmse: Raíz del error cuadrático medio.
            - mae: Error absoluto medio.
            - r2: Coeficiente de determinación.
    
    Example:
        >>> mse, rmse, mae, r2 = calcular_metricas_evaluacion(modelo.predict(X_test), y_test)
        >>> print(f"R²: {r2:.4f}")
    """
    
    mse = mean_squared_error(y_real, y_prediccion)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_real, y_prediccion)
    r2 = r2_score(y_real, y_prediccion)

    if verbose:
        print("\nEvaluación del modelo:")
        print(f"MSE (Error cuadrático medio): {mse:.4f}")
        print(f"RMSE (Raíz del error cuadrático medio): {rmse:.4f}")
        print(f"MAE (Error absoluto medio): {mae:.4f}")
        print(f"R² (Coeficiente de determinación): {r2:.4f}")
        print(f"El modelo explica aproximadamente el {r2:.2%} de la varianza")
    
    return mse, rmse, mae, r2

In [5]:
def plot_matriz_correlacion(df: pd.DataFrame, variable_dependiente:str):
    """Genera una matriz de correlación interactiva utilizando Plotly.
    
    Crea una visualización de mapa de calor que muestra las correlaciones entre todas
    las variables numéricas del DataFrame, con anotaciones de los valores exactos.
    Adicionalmente, imprime un ranking de correlaciones con la variable dependiente
    especificada.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene las variables numéricas para analizar.
        variable_dependiente (str): Nombre de la columna que se considera la variable dependiente,
                                   cuyas correlaciones se mostrarán ordenadas.
    
    Returns:
        None: Muestra el gráfico interactivo con Plotly y una tabla de correlaciones
              con la variable dependiente en la consola.
    
    Example:
        >>> plot_matriz_correlacion(df_calories, "calories")
    """
    
    # Crear matriz de correlación para visualizar relaciones entre variables con Plotly
    correlation_matrix = df.corr()

    # Crear un mapa de calor interactivo con Plotly
    fig = px.imshow(
        correlation_matrix,
        text_auto='.2f',                    # Mostrar valores numéricos con 2 decimales
        color_continuous_scale='RdBu_r',    # Esquema de colores (rojo-blanco-azul invertido)
        zmin=-1, zmax=1,                    # Rango de valores
        aspect="auto",                      # Ajustar aspecto automáticamente
        title='Matriz de Correlación entre Variables'
    )

    # Mejorar el diseño
    fig.update_layout(
        width=800, 
        height=700,
        coloraxis_colorbar=dict(
            title="Coeficiente<br>de Correlación",
            thicknessmode="pixels", thickness=20,
            lenmode="pixels", len=500,
            yanchor="top", y=1,
            ticks="outside"
        ),
        font=dict(size=12),
    )

    # Identificar y mostrar las correlaciones más fuertes con 'calories'
    correlaciones_con_calories = correlation_matrix[variable_dependiente].sort_values(ascending=False)

    # Mostrar el gráfico interactivo
    fig.show(renderer='iframe')

    # Imprimir correlaciones con 'calories'
    print(f"\nCorrelaciones con '{variable_dependiente}':")
    print(correlaciones_con_calories)

## 1. Lectura de datos

El conjunto de datos suministrado contiene información sobre reservas de hoteles hechas a lo largo del tiempo, incluyendo detalles sobre los clientes, el comportamiento de reserva y la probabilidad de cancelación.

El objetivo será predecir si una reserva será cancelada (`is_canceled = 1`) o no (`is_canceled = 0`) aplicando modelos de clasificación binaria.


### Descripción de variables

| Nombre Variable                  | Descripción                                              |
| -------------------------------- | -------------------------------------------------------- |
| `hotel`                          | Tipo de hotel: City Hotel o Resort Hotel                 |
| `is_canceled`                    | Variable objetivo: 1 si fue cancelado, 0 si no           |
| `lead_time`                      | Días entre la reserva y la fecha de llegada              |
| `arrival_date_year`              | Año de llegada                                           |
| `arrival_date_month`             | Mes de llegada                                           |
| `arrival_date_week_number`       | Número de la semana del año                              |
| `arrival_date_day_of_month`      | Día del mes de llegada                                   |
| `stays_in_weekend_nights`        | Noches de fin de semana reservadas                       |
| `stays_in_week_nights`           | Noches entre semana reservadas                           |
| `adults`                         | Número de adultos                                        |
| `children`                       | Número de niños                                          |
| `babies`                         | Número de bebés                                          |
| `meal`                           | Tipo de comida reservada                                 |
| `country`                        | País de origen del cliente                               |
| `market_segment`                 | Canal de marketing (online, offline, grupos...)          |
| `distribution_channel`           | Canal de distribución (directo, TA/TO...)                |
| `is_repeated_guest`              | 1 si el cliente ha estado anteriormente                  |
| `previous_cancellations`         | Nº de cancelaciones anteriores                           |
| `previous_bookings_not_canceled` | Nº de reservas previas no canceladas                     |
| `reserved_room_type`             | Tipo de habitación reservada                             |
| `assigned_room_type`             | Tipo de habitación asignada                              |
| `booking_changes`                | Nº de cambios en la reserva                              |
| `deposit_type`                   | Tipo de depósito: No Deposit, Refundable, etc.           |
| `agent`                          | ID del agente (puede ser nulo)                           |
| `company`                        | ID de la empresa (puede ser nulo)                        |
| `days_in_waiting_list`           | Días en lista de espera                                  |
| `customer_type`                  | Tipo de cliente: Transient, Group, etc.                  |
| `adr`                            | Average Daily Rate (precio promedio por noche)           |
| `required_car_parking_spaces`    | Plazas de parking solicitadas                            |
| `total_of_special_requests`      | Nº de peticiones especiales                              |
| `reservation_status`             | Estado final de la reserva: Check-Out, Canceled, No-Show |
| `reservation_status_date`        | Fecha en que se actualizó el estado                      |

### Carga de datos

In [8]:
# Cargamos el dataset de la práctica final
df = pd.read_csv(PATH_DATASET_PRACTICA_FINAL)

## 2. Comprobación y preprocesamiento de datos
Veremos cuáles son los datos que tenemos en el dataframe, si los tipos asignados tienen sentido, descriptivo estadístico de las columnas, etc

In [9]:
# Mostrar las primeras filas del dataframe
df.head(10)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03
6,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,No Deposit,NaN,NaN,0,Transient,107.0,0,0,Check-Out,2015-07-03
7,Resort Hotel,0,9,2015,July,27,1,0,2,2,...,No Deposit,303.0,NaN,0,Transient,103.0,0,1,Check-Out,2015-07-03
8,Resort Hotel,1,85,2015,July,27,1,0,3,2,...,No Deposit,240.0,NaN,0,Transient,82.0,0,1,Canceled,2015-05-06
9,Resort Hotel,1,75,2015,July,27,1,0,3,2,...,No Deposit,15.0,NaN,0,Transient,105.5,0,0,Canceled,2015-04-22


In [24]:
df['country'].value_counts()

country
PRT    48590
GBR    12129
FRA    10415
ESP     8568
DEU     7287
       ...  
NCL        1
KIR        1
SDN        1
ATF        1
SLE        1
Name: count, Length: 177, dtype: int64

In [10]:
# Mostrar las dimensiones del dataframe
print(f"Dimensiones del dataframe:")
print(f"{df.shape[0]} filas")
print(f"{df.shape[1]} columnas")

Dimensiones del dataframe:
119390 filas
32 columnas


In [11]:
dup_mask = df.duplicated(keep=False)
print(df["is_canceled"].value_counts(normalize=True))
print(df.loc[dup_mask, "is_canceled"].value_counts(normalize=True))
print(df.drop_duplicates()["is_canceled"].value_counts(normalize=True))


is_canceled
0    0.629584
1    0.370416
Name: proportion, dtype: float64
is_canceled
1    0.583867
0    0.416133
Name: proportion, dtype: float64
is_canceled
0    0.725102
1    0.274898
Name: proportion, dtype: float64


In [12]:
# Mostrar la información del dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

---

Primeras observaciones:
- El nombre de las columnas (variables) tiene el formato correcto: minúsculas, sin caracteres especiales salvo barras bajas, sin espacios.
- Hay columnas cuyos tipos de datos no son los esperados. Habrá que investigar si contienen nulos.

---

In [14]:
# Contar valores nulos
df.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

In [15]:
# Verificar si hay valores duplicados en el dataframe
df.duplicated().sum()

np.int64(31994)

In [16]:
# Ratio de registros duplicados frente a totales
print(f"{df.duplicated().sum() / len(df):.2f}")

0.27


### Observaciones

- **Eliminar variables redundantes**. Revisar las columnas ``arrival_date_*``
    - ¿``arrival_date_month`` habría que transformarlo a número de mes (y luego castear a string, porque es categórica)?
    - ¿``arrival_date_week_number`` sería redundante y habría que eliminarla?
- **Existen 31.994 valores duplicados, ¿qué hago con ellos: elimino o mantengo?**.Ç
- ``children``
    - no tiene sentido que sea de tipo float64. Debería ser int.
    - Tiene 4 NaNs.
    - ¿Cambio NaN por ceros o elimino esos 4 registros?
- ``country`` tiene 488 NaN.
- ``is_repeated_guest`` es int64 pero debería ser string. Podría tener NaNs. ¿Sustituyo NaNs por '0' o elimino todo el registro porque no sé cuál valor era realmente de verdad, si 0 o 1?
- ``agent``:
    - es float64 pero debería ser string.
    - podría esconder NaN.
    - ¿Elimino filas con NaN, o sustituyo NaN por '0'? En verdad tengo una ausencia de valor en esas celdas, no tengo certeza de que la realidad fuera '0'; quizás sí había un agente asociado pero se perdió por el camino, por eso, ¿sería mejor eliminar el registro?
    - El ID de agente aparece como string proviniente de float. Antes de castear a string, ¿debería castear a Int64 para que luego los IDs aparezcan sin el '.0' decimal? ¿Quizás debería eliminar simplemente '.0' de cada celda de la columna?
- ``company``:
    - Lo mismo que ``agent``
- ``required_car_parking_spaces``: posible columna casi cte, posible descarte.
- En el Readme.md de los datos de la práctica se lee "ID del agente (puede ser nulo)" y "ID de la empresa (puede ser nulo)". ¿Eso nos debería decir algo acerca de si se deben eliminar los registros sin esos IDs? Quizás podemos convertir la columna a string directamente para que los nulos se conviertan en cadena vacía (**¿equivaldría dejarlo como string vacío a un valor válido para predecir?**).
- ``reservation_status_date`` está congelado en un string. Su información no es utilizable para predecir. Hay que convertirlo a date y crear varias columnas expandidas:
    - ``reservation_status_date_year``
    - ``reservation_status_date_month``
    - ``reservation_status_date_week_number``
    - ``reservation_status_date_day_of_month``
- La cheatsheet de algoritmos recomienda
    - **Escalar** datos
    - **Codificar** categorías

### Columnas con IDs ``agent`` y ``company``

Estas columnas tienen IDs que se hacen pasar por números. Además, presentan registros con valores nulos permitidos (no resultan en errores de importación). Así que conviene retener su información.

- Convertir de float a entero sin decimales.
- Convertir a string.

In [17]:
print(f"Recuento de valores")
print(df['reservation_status_date'].value_counts(dropna=False))

print(f"\nValores únicos")
print(f"Recuento: {df['reservation_status_date'].nunique()}")
print(f"{df['reservation_status_date'].unique()}")

Recuento de valores
reservation_status_date
2015-10-21    1461
2015-07-06     805
2016-11-25     790
2015-01-01     763
2016-01-18     625
              ... 
2015-04-21       1
2015-03-13       1
2015-03-29       1
2015-04-27       1
2015-03-10       1
Name: count, Length: 926, dtype: int64

Valores únicos
Recuento: 926
<StringArray>
['2015-07-01', '2015-07-02', '2015-07-03', '2015-05-06', '2015-04-22',
 '2015-06-23', '2015-07-05', '2015-07-06', '2015-07-07', '2015-07-08',
 ...
 '2015-03-13', '2015-05-05', '2015-03-29', '2015-06-10', '2015-04-27',
 '2014-10-17', '2015-01-20', '2015-02-17', '2015-03-10', '2015-03-23']
Length: 926, dtype: str


### Preprocesamiento de datos

In [18]:
# Fijamos la columna que representa las clases a predecir
target = 'is_canceled'

In [19]:
# La variable objetivo es categórica pero su formato de datos ahora es int64
# Cambiamos el formato de la columna target a string (object) así como el resto de variables categóricas.

dict_astype = {
    target : 'str',
    'is_repeated_guest': 'str',
    'agent': 'str',
    'company': 'str'
}

df_preprocessed = df.astype(dict_astype)

In [20]:
# Reemplazar los valores numéricos de la columna 'target' por los nombres de las clases
dict_target = {
    '0' : 'No cancelado',
    '1' : 'Cancelado'
}

df_preprocessed[target] = df_preprocessed[target].replace(target)

ValueError: Series.replace must specify either 'value', a dict-like 'to_replace', or dict-like 'regex'.

In [21]:
# Verificamos los tipos del dataFrame
df_preprocessed.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  str    
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [22]:
# Descripción estadística de las columnas numéricas
df_preprocessed.describe(include='number').transpose()

,count,mean,std,min,25%,50%,75%,max
lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
arrival_date_year,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0
arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0
previous_cancellations,119390.0,0.087118,0.844336,0.00,0.00,0.000,0.0,26.0


In [23]:
# Descripción estadística de las columnas categóricas
df_preprocessed.describe(include='object').transpose()

C:\Users\Jaime\AppData\Local\Temp\ipykernel_20636\3799974276.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_preprocessed.describe(include='object').transpose()


,count,unique,top,freq
hotel,119390,2,City Hotel,79330
is_canceled,119390,2,0,75166
arrival_date_month,119390,12,August,13877
meal,119390,5,BB,92310
country,118902,177,PRT,48590
market_segment,119390,8,Online TA,56477
distribution_channel,119390,5,TA/TO,97870
is_repeated_guest,119390,2,0,115580
reserved_room_type,119390,10,A,85994
assigned_room_type,119390,12,A,74053


In [ ]:
print(df_preprocessed['agent'])

print(f"\n\nEl ID de agente aparece como string proviniente de float. Antes de castear a string, ¿debería castear a Int64 para que luego los IDs aparezcan sin el '.0' decimal? ¿Quizás debería eliminar simplemente '.0' de cada celda de la columna?")

0           nan
1           nan
2           nan
3         304.0
4         240.0
          ...  
119385    394.0
119386      9.0
119387      9.0
119388     89.0
119389      9.0
Name: agent, Length: 119390, dtype: object


El ID de agente aparece como string proviniente de float. Antes de castear a string, ¿debería castear a Int64 para que luego los IDs aparezcan sin el '.0' decimal? ¿Quizás debería eliminar simplemente '.0' de cada celda de la columna?


In [ ]:
# Obtener las frecuencias absolutas de la columna target
df_preprocessed[target].value_counts()

is_canceled
0    75166
1    44224
Name: count, dtype: int64

In [ ]:
# Obtener las frecuencias absolutas de la columna target
df_preprocessed[target].value_counts(normalize=True)

is_canceled
0    0.629584
1    0.370416
Name: proportion, dtype: float64

In [ ]:
columnas_numericas = df_preprocessed.select_dtypes(include='number').columns.tolist()

In [ ]:
columnas_numericas_con_target = df_preprocessed.select_dtypes(include='number').columns.tolist()
if target not in columnas_numericas_con_target:
    columnas_numericas_con_target.append(target)

print(columnas_numericas_con_target)

['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'is_canceled']


In [ ]:
%%time

# Visualización de las variables numéricas con un gráfico de dispersión distinguiendo por la clase
sns.pairplot(df_preprocessed[columnas_numericas_con_target], hue='is_canceled', palette='Set1')

CPU times: total: 0 ns
Wall time: 0 ns
